In [ ]:
# 1. Список источников
easy_sources = [
    'https://investfunds.ru/', # спарсил ооочень задолго, с 2014
    'https://smart-lab.ru/', # спарсил с конца 2024 по начало 2026
    'https://www.kommersant.ru/', # спарсил на 430 дней назад - до ноября 2024
    'https://www.interfax.ru/business/', # спарсил на 430 дней назад также
    'https://www.comnews.ru/', # спарсил все 74 страницы, которые были
]
today = [

]

medium_sources = [
    'https://ru.tradingview.com/markets/stocks-russia/', # сложненько
    'https://ru.investing.com/equities/russia',
    'https://www.moex.com/ru/news/', # сложненько
]

hard_sources = [
    'https://bcs-express.ru/category',
    'https://www.finam.ru/', # сложна
    'https://ria.ru/economy', # два раза не пошло
    'https://dzen.ru/news/rubric/business', # прокрут по JS
    'https://www.rbc.ru/',# есть rss, можно получить данные за сегодня, не за сегодня  сложна, тут

]

#blacklist
# 'https://www.tbank.ru/invest/pulse/', # решил пока не брать
# 'https://gazprombank.investments/', # решил пока не брать
# 'https://conomy.ru/news', # последняя новость 1.5 года назад - решил не брать

# Smartlab - работает


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time
import random


HEADERS = {"User-Agent": "Mozilla/5.0"}


In [ ]:
r = requests.get('https://smart-lab.ru/news/date/2026-01-20', headers=HEADERS, timeout=10)

In [ ]:
r.text

'<!DOCTYPE html><html lang="ru"><head>\n\t<!-- Global Site Tag (gtag.js) - Google Analytics -->\n\t<script async src="https://www.googletagmanager.com/gtag/js?id=UA-16537214-3"></script>\n\t<script>\n\twindow.dataLayer = window.dataLayer || [];\n\tfunction gtag(){dataLayer.push(arguments);}\n\tgtag(\'js\', new Date());\n\tgtag(\'config\', \'UA-16537214-3\', {\n\t\t\t\'custom_map\': {\n\t\t\t\t\'dimension1\' : \'user_registred\',\n\t\t\t\t\'dimension2\' : \'content_owner\'\n\n\t\t\t},\n\n\t\t\t\'user_registred\': \'No\',\n\t\t\t\'content_owner\': \'No\'\t});\n\t</script>\n\t<meta name="push-subscribes" content="no"><title>Новости компаний и новости по акциям за дату 20.01.2026</title><meta http-equiv="content-type" content="text/html; charset=utf-8"/><link rel="manifest" href="/manifest.json"><meta name="DESCRIPTION" content="Все новости по компаниям, которые торгуются на Московской бирже. Причины роста и падения акций. Объяснение новостей экспертами. Дата 20.01.2026"/><meta name="KEYWO

In [ ]:
url = "https://smart-lab.ru/news/date/2026-01-20/"
resp = requests.get(url)
soup = BeautifulSoup(resp.text, 'html.parser')

news_items = soup.find_all('h3', class_='feed title bluid_48504')

for item in news_items:
    date = item.find('span', class_='user').text.strip()
    link = item.find('a')['href']
    title = item.find('a')['title']
    print(date, title, "https://smart-lab.ru" + link)


20/01 Трамп: пока не удается договориться, чтобы Украина и Россия одновременно проявили готовность к урегулированию конфликта https://smart-lab.ru/blog/1255548.php
20/01 📈 Акции Селигдара сегодня прибавляют 7,3% на фоне рекордных цен на золото https://smart-lab.ru/blog/1255547.php
20/01 Трамп утверждает, что лучше управляет американской экономикой, чем Уоррен Баффетт своим бизнесом https://smart-lab.ru/blog/1255545.php
20/01 Трамп: США придется вернуть полученные от пошлин доходы, если Верховный суд страны признает незаконными действия американских властей в тарифной политике https://smart-lab.ru/blog/1255541.php
20/01 📉 Dow Jones -1,72%, S&P 500 -1,93%, Nasdaq -2,18%. Американский рынок падает второй день из-за конфликта Трампа с Европой https://smart-lab.ru/blog/1255539.php
20/01 Дмитриев заявил, что его встречи в Давосе проходят конструктивно. Он также отметил, что все больше и больше людей осознают правильность позиции России https://smart-lab.ru/blog/1255538.php
20/01 ЭсЭфАй: на д

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Функция для получения новостей за одну страницу
def get_news_for_page(day_url):
    resp = requests.get(day_url, headers=headers)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    news_items = soup.find_all('h3', class_='feed title bluid_48504')
    news_list = []

    for item in news_items:
        date_span = item.find('span', class_='user')
        link_tag = item.find('a')

        if date_span and link_tag:
            news_date = date_span.text.strip()
            title = link_tag['title'].strip()
            link = "https://smart-lab.ru" + link_tag['href']
            news_list.append({
                'date': news_date,
                'title': title,
                'link': link
            })
    return news_list




In [ ]:
# Основной цикл по дням
start_date = date(2024, 1, 1)
end_date = date(2026, 1, 20)

all_news = []

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime('%Y-%m-%d')
    print(f"Сбор новостей за {date_str}...")

    # Лимитируем до 5 страниц
    for page in range(1, 6):
        time.sleep(random.uniform(0.5, 1.5))
        if page == 1:
            url = f"https://smart-lab.ru/news/date/{date_str}/"
        else:
            url = f"https://smart-lab.ru/news/date/{date_str}/page{page}/"

        try:
            news_on_page = get_news_for_page(url)
            if not news_on_page:
                break  # Если на странице нет новостей, выходим
            all_news.extend(news_on_page)
        except Exception as e:
            print(f"Ошибка при запросе {url}: {e}")
            break

    current_date += timedelta(days=1)


Сбор новостей за 2024-01-01...
Сбор новостей за 2024-01-02...
Сбор новостей за 2024-01-03...
Сбор новостей за 2024-01-04...


KeyboardInterrupt: 

In [ ]:

# Создаем DataFrame
df = pd.DataFrame(all_news)
print(df.head(), df.tail())

In [ ]:
# При желании можно сохранить в CSV
# df.to_csv('/content/drive/MyDrive/ВКР/data/news/smartlab_news_2024-2026.csv', index=False)

# Investfunds попытка №2 - работает


In [ ]:
import requests
from bs4 import BeautifulSoup

base_url = "https://investfunds.ru/news/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Content-Type': 'application/x-www-form-urlencoded'
}

results = []

max_pages = 880
for page in range(1, max_pages + 1):
    r = requests.post(f'{base_url}?limit=50&page={page}', headers=headers)
    r.encoding = 'utf-8'
    soup = BeautifulSoup(r.text, 'html.parser')

    news_list = soup.select('ul.news_list li')
    if not news_list:
        print(f"Новости не найдены на странице {page}")
        continue

    current_date = ""  # будем хранить текущую дату
    for item in news_list:
        # если это дата
        if "date" in item.get("class", []):
            current_date = item.get_text(strip=True)
            continue  # идём к следующему элементу

        # если это новость
        if "item" in item.get("class", []):
            title_tag = item.select_one('div.lnk a.indent_right_10')
            link = title_tag['href'] if title_tag else ''
            title = title_tag.get_text(strip=True) if title_tag else ''
            time_tag = item.select_one('span.time')
            time = time_tag.get_text(strip=True) if time_tag else ''
            source_tag = item.select_one('div.lnk a.source')
            source = source_tag.get_text(strip=True) if source_tag else ''

            results.append({
                'date': current_date,
                'time': time,
                'title': title,
                'link': 'https://investfunds.ru' + link if link.startswith("/") else link,
                'source': source
            })

for n in results:
    print(n)


In [ ]:
# собираем DataFrame
df = pd.DataFrame(results)

# сохраняем в CSV
# df.to_csv('/content/drive/MyDrive/ВКР/data/news/investfunds_news.csv', index=False, encoding='utf-8-sig')
# print("Сбор новостей завершен. Всего новостей:", len(df))

# Коммерсантъ

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import random

rubrics = {
    "Экономика": 3,
    "Бизнес": 4,
    "Финансы": 40,
    "Потребительский рынок": 41
}

days_to_fetch = 430  # сколько дней назад парсить
headers = {"User-Agent": "Mozilla/5.0"}

all_news = []

for rubric_name, rubric_id in rubrics.items():
    print(f"Парсим рубрику: {rubric_name}")
    for i in range(days_to_fetch):
        date = datetime.today() - timedelta(days=i)
        date_str = date.strftime("%Y-%m-%d")
        url = f"https://www.kommersant.ru/archive/rubric/{rubric_id}/day/{date_str}"

        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Не удалось получить страницу {url}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        articles = soup.select("article.rubric_lenta__item")
        print(f"Найдено {len(articles)} новостей за {date_str}")

        for art in articles:
            news = {
                "rubric": rubric_name,
                "date": art.get("data-article-date", date_str),  # иногда атрибут нет
                "time": art.select_one("p.rubric_lenta__item_tag").get_text(strip=True).split(", ")[-1]
                        if art.select_one("p.rubric_lenta__item_tag") else "",
                "title": art.get("data-article-title", "").strip(),
                "link": art.get("data-article-url", "").strip(),
                "description": art.get("data-article-description", "").strip()
            }
            all_news.append(news)
        # задержка 1 сек, чтобы сайт не забанил
        # time.sleep(random.uniform(0.5, 2))

print(f"Всего новостей собрано: {len(all_news)}")

# пример первых 5 новостей
for news in all_news[:5]:
    print(news)


In [ ]:
all_news[:5], all_news[-5:]

In [ ]:
import pandas as pd

# создаём DataFrame из списка словарей
df = pd.DataFrame(all_news)

print(df.head())

In [ ]:
# сохраняем в CSV (UTF-8, чтобы кириллица не слетела)
# df.to_csv("/content/drive/MyDrive/ВКР/data/news/kommersant_news.csv", index=False, sep=";")


# Interfax

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

headers = {"User-Agent": "Mozilla/5.0"}
all_news = []

# Сколько дней назад парсить
days_to_fetch = 430

for i in range(days_to_fetch):
    date = datetime.today() - timedelta(days=i)
    date_str = date.strftime("%Y/%m/%d")  # для URL Interfax
    url = f"https://www.interfax.ru/business/news/{date_str}/"

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Не удалось получить страницу {url}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    news_blocks = soup.select("div.an > div[data-id]")
    print(f"Найдено {len(news_blocks)} новостей за {date.strftime('%Y-%m-%d')}")

    for nb in news_blocks:
        time = nb.find("span").get_text(strip=True)
        link_tag = nb.find("a")
        title_tag = nb.find("h3")

        news = {
            "site": "interfax.ru",
            "date": date.strftime("%Y-%m-%d"),
            "time": time,
            "title": title_tag.get_text(strip=True) if title_tag else "",
            "link": f"https://www.interfax.ru{link_tag['href']}" if link_tag else "",
            "description": "",  # тут пока нет описания
        }
        all_news.append(news)

print(f"Всего новостей собрано: {len(all_news)}")
for news in all_news[:5]:
    print(news)

In [ ]:
for news in all_news[-5:]:
    print(news)

In [ ]:
import pandas as pd

# all_news — список словарей, который мы получили ранее
# пример словаря:
# {"site": "interfax.ru", "rubric": "Экономика", "date": "2026-01-27", "time": "23:48", "title": "Заголовок", "link": "https://...", "description": ""}

# Формируем DataFrame
df = pd.DataFrame(all_news)

# Проверяем первые 5 строк
print(df.head())




In [ ]:
# сохраняем
# df.to_csv("/content/drive/MyDrive/ВКР/data/news/interfax_news.csv", index=False, encoding="utf-8-sig")  # utf-8-sig, чтобы Excel корректно открыл


# Comnews

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

base_url = "https://www.comnews.ru/investments"
headers = {"User-Agent": "Mozilla/5.0"}

all_news = []

# Пример: парсим первые 2 страницы
for page in range(0, 74):  # page=0 — первая страница, page=1 — вторая и т.д. - всего там 74
    url = f"{base_url}?page={page}" if page > 0 else base_url
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Ошибка доступа к {url}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    nodes = soup.select("div.node")  # все новости

    for node in nodes:
        date = node.select_one("div.date")
        title_tag = node.select_one("div.title a")
        body_tag = node.select_one("div.body a")

        news = {
            "date": date.get_text(strip=True) if date else "",
            "title": title_tag.get_text(strip=True) if title_tag else "",
            "link": "https://www.comnews.ru" + title_tag['href'] if title_tag else "",
            "description": body_tag.get_text(strip=True) if body_tag else ""
        }
        all_news.append(news)


In [ ]:
for i in all_news[-5:]:
  print(i)

In [ ]:
# Формируем DataFrame
df = pd.DataFrame(all_news)
print(df.head())

In [ ]:
len(df)

In [ ]:
# Сохраняем в CSV
# df.to_csv("/content/drive/MyDrive/ВКР/data/news/comnews_news.csv", index=False, encoding="utf-8-sig")

# Единый датасет новостей

In [ ]:
COLUMNS = [
    "source",        # сайт
    "date",          # datetime (дата + время)
    "date_only",     # YYYY-MM-DD (удобно для агрегаций)
    "time",          # HH:MM или None
    "title",
    "link",
    "description",
    "rubric"
]

In [ ]:
import pandas as pd

class BaseNewsLoader:
    source = None

    def load(self, path: str) -> pd.DataFrame:
        df = pd.read_csv(path)
        df = self.normalize(df)
        df["source"] = self.source
        return df

    def normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError

In [ ]:
class SmartLabLoader(BaseNewsLoader):
    source = "smart-lab.ru"

    def normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # источник
        df["source"] = self.source

        # дата
        df["date_only"] = df["date"]
        df["date"] = pd.to_datetime(
            df["date_only"],
            format="%d/%m/%y",
            errors="coerce"
        )

        # обязательные, но отсутствующие поля
        df["time"] = None
        df["description"] = None
        df["rubric"] = None

        return df[[
            "source",
            "date",
            "date_only",
            "time",
            "title",
            "link",
            "description",
            "rubric"
        ]]


In [ ]:
import pandas as pd

class InvestFundsLoader(BaseNewsLoader):
    source = "investfunds.ru"

    def normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # источник
        df["source"] = self.source

        # rubric — реальный первоисточник новости
        df["rubric"] = df["source_y"] if "source_y" in df.columns else df["source"]

        # текущая дата для "Сегодня"
        today = pd.Timestamp.today().normalize()

        # словарь для замены русских месяцев на числа
        months = {
            "января": "01",
            "февраля": "02",
            "марта": "03",
            "апреля": "04",
            "мая": "05",
            "июня": "06",
            "июля": "07",
            "августа": "08",
            "сентября": "09",
            "октября": "10",
            "ноября": "11",
            "декабря": "12"
        }

        # функция для парсинга даты
        def parse_date(x):
            x = str(x).strip()
            if x.lower() == "сегодня":
                return today
            # заменяем месяц на число
            for month, num in months.items():
                if month in x:
                    x = x.replace(month, num)
                    break
            # теперь формат "23 01 2026" → "%d %m %Y"
            try:
                return pd.to_datetime(x, format="%d %m %Y", errors="coerce")
            except:
                return pd.NaT

        df["date_only"] = df["date"].apply(parse_date)

        # если есть время, добавляем его к дате
        df["time"] = df["time"].fillna("00:00")
        df["date"] = pd.to_datetime(
            df["date_only"].dt.strftime("%Y-%m-%d") + " " + df["time"],
            errors="coerce"
        )

        # описание отсутствует
        df["description"] = None

        return df[[
            "source",
            "date",
            "date_only",
            "time",
            "title",
            "link",
            "description",
            "rubric"
        ]]


In [ ]:
class KommersantLoader(BaseNewsLoader):
    source = "kommersant.ru"

    def load(self, path: str) -> pd.DataFrame:
        df = pd.read_csv(path, sep=';')
        df = self.normalize(df)
        df["source"] = self.source
        return df

    def normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Источник
        df["source"] = self.source

        # Приведение даты и времени к datetime
        df["date"] = pd.to_datetime(
            df["date"] + " " + df["time"],
            errors="coerce",
            dayfirst=True
        )

        # Выделяем только дату (без времени)
        df["date_only"] = df["date"].dt.date.astype(str)

        # description может быть пустым, оставляем как есть
        if "description" not in df.columns:
            df["description"] = None

        # rubric оставляем из исходного df
        if "rubric" not in df.columns:
            df["rubric"] = None

        # Собираем в стандартный порядок колонок
        return df[[
            "source",
            "date",
            "date_only",
            "time",
            "title",
            "link",
            "description",
            "rubric"
        ]]


In [ ]:
class InterfaxLoader(BaseNewsLoader):
    source = "interfax.ru"

    def normalize(self, df):
        df['source'] = 'interfax.ru'
        # гарантируем строки
        df["date"] = df["date"].astype(str)
        df["time"] = df["time"].astype(str)

        # datetime
        df["date"] = pd.to_datetime(
            df["date"] + " " + df["time"],
            errors="coerce"
        )

        df["date_only"] = df["date"].dt.date.astype(str)

        if "description" not in df.columns:
            df["description"] = None

        df["rubric"] = "Бизнес"  # т.к. парсили /business/

        return df[[
            "source",
            "date",
            "date_only",
            "time",
            "title",
            "link",
            "description",
            "rubric"
        ]]


In [ ]:
class ComNewsLoader(BaseNewsLoader):
    source = "comnews.ru"

    def normalize(self, df):
        df['source'] = 'comnews.ru'
        # Преобразуем дату
        df["date"] = pd.to_datetime(
            df["date"],
            format="%d.%m.%Y",
            errors="coerce"
        )

        # Только дата
        df["date_only"] = df["date"].dt.date.astype(str)

        # Время нет
        df["time"] = None

        # Рубрика
        df["rubric"] = "Инвестиции"

        # Описание: убираем лишние переносы и пробелы
        if "description" in df.columns:
            df["description"] = df["description"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
        else:
            df["description"] = None

        return df[[
            "source",
            "date",
            "date_only",
            "time",
            "title",
            "link",
            "description",
            "rubric"
        ]]


In [ ]:
loaders = [
    SmartLabLoader(),
    InvestFundsLoader(),
    KommersantLoader(),
    InterfaxLoader(),
    ComNewsLoader()
]

paths = [
    "/content/drive/MyDrive/ВКР/data/news/smartlab_news_2024-2026.csv",
    "/content/drive/MyDrive/ВКР/data/news/investfunds_news.csv",
    "/content/drive/MyDrive/ВКР/data/news/kommersant_news.csv",
    "/content/drive/MyDrive/ВКР/data/news/interfax_news.csv",
    "/content/drive/MyDrive/ВКР/data/news/comnews_news.csv"
]

dfs = []

for loader, path in zip(loaders, paths):
    try:
        print(f"Загружаем {loader.source} из {path}...")
        df = loader.load(path)
        # убираем лишние пробелы в названиях колонок
        df.columns = df.columns.str.strip()
        dfs.append(df)
        print(f"Загружено {len(df)} новостей из {loader.source}")
    except Exception as e:
        print(f"Ошибка при загрузке {path}: {e}")

# объединяем все DataFrame
all_news = pd.concat(dfs, ignore_index=True)

# сортировка по дате
all_news = all_news.sort_values("date")

# удаляем дубликаты по title и link
all_news = all_news.drop_duplicates(subset=["title", "link"])

# сохраняем итог
# all_news.to_csv(
#     "/content/drive/MyDrive/ВКР/data/all_news.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

print("Итоговый размер объединённого датасета:", len(all_news))


Загружаем smart-lab.ru из /content/drive/MyDrive/ВКР/data/news/smartlab_news_2024-2026.csv...
Загружено 4049 новостей из smart-lab.ru
Загружаем investfunds.ru из /content/drive/MyDrive/ВКР/data/news/investfunds_news.csv...
Загружено 42742 новостей из investfunds.ru
Загружаем kommersant.ru из /content/drive/MyDrive/ВКР/data/news/kommersant_news.csv...
Загружено 17020 новостей из kommersant.ru
Загружаем interfax.ru из /content/drive/MyDrive/ВКР/data/news/interfax_news.csv...


/tmp/ipython-input-93298606.py:17: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(


Загружено 21598 новостей из interfax.ru
Загружаем comnews.ru из /content/drive/MyDrive/ВКР/data/news/comnews_news.csv...
Загружено 2205 новостей из comnews.ru
Итоговый размер объединённого датасета: 86598


In [ ]:
all_news['source'].value_counts()

,count
source,
investfunds.ru,42742
interfax.ru,21598
kommersant.ru,16635
smart-lab.ru,3418
comnews.ru,2205


In [ ]:
all_news[all_news['source'] == 'investfunds.ru'][-5:]# дату починить

,source,date,date_only,time,title,link,description,rubric
4053,investfunds.ru,2026-01-22 10:31:00,2026-01-22 00:00:00,10:31,Венчурный фонд Т-банка инвестировал в разработ...,https://investfunds.ru/news/171723/,None,investfunds.ru
4052,investfunds.ru,2026-01-23 12:03:00,2026-01-23 00:00:00,12:03,Какие результаты могут показать инвестфонды в ...,https://investfunds.ru/news/171725/,None,investfunds.ru
4051,investfunds.ru,2026-01-23 15:34:00,2026-01-23 00:00:00,15:34,Инвестиционные тренды-2026: как выстраивать по...,https://investfunds.ru/news/171727/,None,investfunds.ru
4050,investfunds.ru,2026-01-28 10:13:00,2026-01-28 00:00:00,10:13,Группа «Эталон» объявляет о начале сбора заяво...,https://investfunds.ru/news/171729/,None,investfunds.ru
4049,investfunds.ru,2026-01-28 11:09:00,2026-01-28 00:00:00,11:09,Ноябрь 2025 г.: розничные инвесторы увеличиваю...,https://investfunds.ru/news/171731/,None,investfunds.ru


In [ ]:
all_news[all_news['source'] == 'interfax.ru'][-5:]# OK

,source,date,date_only,time,title,link,description,rubric
63815,interfax.ru,2026-01-27 09:44:00,2026-01-27,09:44,"""Ъ"" сообщил о предложении Минфина легализовать...",https://www.interfax.ru/business/1069630,NaN,Бизнес
63814,interfax.ru,2026-01-27 09:50:00,2026-01-27,09:50,Китайская Anta Sports покупает 29% акций Puma ...,https://www.interfax.ru/business/1069634,NaN,Бизнес
63813,interfax.ru,2026-01-27 10:08:00,2026-01-27,10:08,Рынок акций открылся в основную сессию незначи...,https://www.interfax.ru/business/1069639,NaN,Бизнес
63812,interfax.ru,2026-01-27 10:09:00,2026-01-27,10:09,Рубль утром дешевеет в паре с юанем на фоне не...,https://www.interfax.ru/business/1069640,NaN,Бизнес
63811,interfax.ru,2026-01-27 12:06:00,2026-01-27,12:06,"Опрос ЦБ показал, что инфляционные ожидания на...",https://www.interfax.ru/business/1069659,NaN,Бизнес


In [ ]:
all_news[all_news['source'] == 'kommersant.ru'][-5:]# ОК

,source,date,date_only,time,title,link,description,rubric
59411,kommersant.ru,2026-01-27 09:15:00,2026-01-27,09:15,Продажи средств от похмелья выросли после Ново...,https://www.kommersant.ru/doc/8378496,NaN,Потребительский рынок
50180,kommersant.ru,2026-01-27 10:06:00,2026-01-27,10:06,Аэропорты Внуково и Жуковский сообщили о штатн...,https://www.kommersant.ru/doc/8378689,NaN,Бизнес
50179,kommersant.ru,2026-01-27 10:18:00,2026-01-27,10:18,Прокуратура начала проверку в аэропорту Шереме...,https://www.kommersant.ru/doc/8378713,NaN,Бизнес
50178,kommersant.ru,2026-01-27 11:03:00,2026-01-27,11:03,Более 15 рейсов из Пулково в Москву задержали ...,https://www.kommersant.ru/doc/8378764,NaN,Бизнес
50177,kommersant.ru,2026-01-27 11:39:00,2026-01-27,11:39,«Победа» перевела часть рейсов из Шереметьево ...,https://www.kommersant.ru/doc/8378797,NaN,Бизнес


In [ ]:
all_news[all_news['source'] == 'smart-lab.ru'][-140:-130]# починить дату без года(новый год)

,source,date,date_only,time,title,link,description,rubric
3906,smart-lab.ru,2025-12-29,29/12/25,None,"Трамп: Не знаю, что Россия заявляла о предпола...",https://smart-lab.ru/blog/1248698.php,None,None
3910,smart-lab.ru,2025-12-30,30/12/25,None,США намерены тщательно изучить обстоятельства ...,https://smart-lab.ru/blog/1249200.php,None,None
3911,smart-lab.ru,2025-12-30,30/12/25,None,Протокол заседания ФРС: большинство участников...,https://smart-lab.ru/blog/1249194.php,None,None
3914,smart-lab.ru,2025-12-30,30/12/25,None,Рост ВВП РФ в III квартале 2025 года составил ...,https://smart-lab.ru/blog/1249135.php,None,None
3913,smart-lab.ru,2025-12-30,30/12/25,None,Скидки на российскую нефть Urals в российских ...,https://smart-lab.ru/blog/1249138.php,None,None
3912,smart-lab.ru,2025-12-30,30/12/25,None,"Зеленский: Лидеры Украины, США и Европы могут ...",https://smart-lab.ru/blog/1249142.php,None,None
3915,smart-lab.ru,NaT,05/01,None,Вице-президент Венесуэлы Делси Родригес официа...,https://smart-lab.ru/blog/1250360.php,None,None
3916,smart-lab.ru,NaT,05/01,None,"Госдепартамент США официально заявил, что Запа...",https://smart-lab.ru/blog/1250354.php,None,None
3917,smart-lab.ru,NaT,05/01,None,На этой неделе США проведут встречу с руководи...,https://smart-lab.ru/blog/1250351.php,None,None
3918,smart-lab.ru,NaT,05/01,None,Мадуро заявил о своей невиновности в суде Нью-...,https://smart-lab.ru/blog/1250341.php,None,None


In [ ]:
all_news[all_news['source'] == 'comnews.ru'][-5:]# время допарсить(если оно там есть в принципе)

,source,date,date_only,time,title,link,description,rubric
85413,comnews.ru,2026-01-23,2026-01-23,None,"Процессная аналитика дала ""Сберу"" финансовый э...",https://www.comnews.ru/content/243391/2026-01-...,Финансовый эффект от процессной аналитики (Pro...,Инвестиции
85412,comnews.ru,2026-01-23,2026-01-23,None,"Сбер приобрел долю в ПАО ""Элемент""",https://www.comnews.ru/content/243404/2026-01-...,"Сбер закрыл сделку по покупке 41,9% акций в ПА...",Инвестиции
85410,comnews.ru,2026-01-26,2026-01-26,None,Криптопаузе быть… или не быть,https://www.comnews.ru/content/243421/2026-01-...,ЛДПР предложила предоставить действующим участ...,Инвестиции
85411,comnews.ru,2026-01-26,2026-01-26,None,WAF и Anti-DDoS в России ускоряют темпы роста,https://www.comnews.ru/content/243419/2026-01-...,В 2025 г. объем потребления WAF и Anti-DDoS в ...,Инвестиции
85409,comnews.ru,2026-01-27,2026-01-27,None,"""Софтлайн"" выпустит облигации на 3 млрд руб.",https://www.comnews.ru/content/243439/2026-01-...,"ПАО ""Софтлайн"" планирует выпустить биржевые об...",Инвестиции
